# Sumativa 2 — Base de resultados y simulación Monte Carlo

**Equipo:** Hernán Saavedra, Lenin Salas y Rodrigo Gómez.  
**Responsable de este bloque:** Hernán Saavedra.

## Objetivo

Preparar resultados de referencia de la Sumativa 1 y simular escenarios
de salida utilizando parámetros estimados en ese análisis.

Se utiliza el dataset IBM HR original completo, sin faltantes artificiales.
Los datos son sintéticos: los resultados no representan predicciones
para una empresa real ni permiten establecer causalidad.

In [1]:
from pathlib import Path
import hashlib
import json
import sys

import numpy as np
import pandas as pd
import scipy
from scipy import stats
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

# Buscar la raíz desde cualquier carpeta dentro del repositorio.
nombre_csv = "WA_Fn-UseC_-HR-Employee-Attrition.csv"
carpeta_actual = Path.cwd().resolve()

raiz = next(
    (
        carpeta
        for carpeta in [carpeta_actual, *carpeta_actual.parents]
        if (carpeta / "datos" / nombre_csv).is_file()
    ),
    None
)

if raiz is None:
    raise FileNotFoundError(
        "No se encontró el dataset en la carpeta datos del repositorio."
    )

ruta_datos = raiz / "datos" / nombre_csv
ruta_notebook_s1 = raiz / "S1" / "Sumativa1_IBM_HR.ipynb"

if not ruta_notebook_s1.is_file():
    raise FileNotFoundError(
        f"No se encontró el notebook de referencia: {ruta_notebook_s1}"
    )

carpeta_figuras = raiz / "S2" / "figuras"
carpeta_resultados = raiz / "S2" / "resultados"

carpeta_figuras.mkdir(parents=True, exist_ok=True)
carpeta_resultados.mkdir(parents=True, exist_ok=True)

# Cargar el original y conservar una copia para el análisis.
original = pd.read_csv(ruta_datos)
df = original.copy(deep=True)

# Configuración de reproducibilidad.
SEMILLA = 42
N_ITERACIONES = 10_000

# Se crearán generadores independientes para cada procedimiento.
# Esto evita depender del orden de ejecución de otros bloques.
sns.set_theme(style="whitegrid")

huella_csv = hashlib.sha256(ruta_datos.read_bytes()).hexdigest()
huella_notebook_s1 = hashlib.sha256(
    ruta_notebook_s1.read_bytes()
).hexdigest()

versiones = {
    "Python": sys.version.split()[0],
    "pandas": pd.__version__,
    "NumPy": np.__version__,
    "SciPy": scipy.__version__,
    "Matplotlib": matplotlib.__version__,
    "seaborn": sns.__version__
}

print("Raíz del repositorio:", raiz)
print(f"Dataset: {df.shape[0]} registros y {df.shape[1]} columnas")
print("Faltantes:", int(df.isna().sum().sum()))
print("Filas duplicadas:", int(df.duplicated().sum()))
print("Semilla base:", SEMILLA)
print("Iteraciones previstas:", N_ITERACIONES)
print("SHA-256 del CSV:", huella_csv)

print("\nVersiones del entorno")
for biblioteca, version in versiones.items():
    print(f"{biblioteca}: {version}")

display(df.head())

Raíz del repositorio: C:\Users\Papá\Documents\2026\Magister\estadisticaComputacional
Dataset: 1470 registros y 35 columnas
Faltantes: 0
Filas duplicadas: 0
Semilla base: 42
Iteraciones previstas: 10000
SHA-256 del CSV: a5c31e38bd7fafc9bc333884eb181b06b41b8e5e488e8f7ccb27199fb3be7659

Versiones del entorno
Python: 3.11.9
pandas: 3.0.6
NumPy: 2.4.6
SciPy: 1.17.1
Matplotlib: 3.11.2
seaborn: 0.13.2


,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,...,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,...,1,80,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,...,4,80,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,...,2,80,0,7,3,3,0,0,0,0
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,1,5,...,3,80,0,8,3,3,8,7,3,0
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,7,...,4,80,1,6,3,3,2,2,2,2


## 1. Resultados de referencia de la Sumativa 1

Se reproducen las medias y los intervalos t del 95 % para ingreso mensual,
edad y antigüedad, utilizando el mismo dataset completo y procedimiento
de la Sumativa 1.

Los resultados se exportan sin redondear. El formato con coma decimal
se aplica únicamente a la presentación.

In [2]:
variables_medias = ["MonthlyIncome", "Age", "YearsAtCompany"]

referencias_medias = []

for variable in variables_medias:
    valores = df[variable].dropna()
    n = len(valores)
    media = valores.mean()
    desviacion = valores.std(ddof=1)
    error_estandar = stats.sem(valores)

    inferior, superior = stats.t.interval(
        confidence=0.95,
        df=n - 1,
        loc=media,
        scale=error_estandar
    )

    referencias_medias.append({
        "variable": variable,
        "n": n,
        "media": media,
        "desviacion_estandar": desviacion,
        "error_estandar": error_estandar,
        "ic95_inferior": inferior,
        "ic95_superior": superior,
        "metodo": "t de Student"
    })

referencias_medias = pd.DataFrame(referencias_medias)

# Mostrar una copia formateada.
presentacion = referencias_medias.copy()

columnas_decimales = [
    "media",
    "desviacion_estandar",
    "error_estandar",
    "ic95_inferior",
    "ic95_superior"
]

for columna in columnas_decimales:
    presentacion[columna] = presentacion[columna].map(
        lambda valor: f"{valor:.2f}".replace(".", ",")
    )

display(presentacion)

# Exportar valores numéricos con precisión suficiente para recuperarlos.
ruta_medias = carpeta_resultados / "S1_referencia_medias.csv"

referencias_medias.to_csv(
    ruta_medias,
    index=False,
    encoding="utf-8-sig",
    float_format="%.17g"
)

print("Referencia guardada:", ruta_medias)

,variable,n,media,desviacion_estandar,error_estandar,ic95_inferior,ic95_superior,metodo
0,MonthlyIncome,1470,"6502,93","4707,96","122,79","6262,06","6743,80",t de Student
1,Age,1470,"36,92","9,14","0,24","36,46","37,39",t de Student
2,YearsAtCompany,1470,"7,01","6,13","0,16","6,69","7,32",t de Student


Referencia guardada: C:\Users\Papá\Documents\2026\Magister\estadisticaComputacional\S2\resultados\S1_referencia_medias.csv


## 2. Correlaciones de referencia de la Sumativa 1

Se reproducen cuatro correlaciones de Spearman previamente examinadas:
ingreso con experiencia, edad con experiencia, antigüedad con experiencia
y distancia con ingreso.

La selección incluye asociaciones positivas y una cercana a cero.
Su estabilidad se evaluará posteriormente mediante remuestreo de filas
completas, conservando el emparejamiento de las variables.

In [3]:
pares_correlaciones = [
    ("MonthlyIncome", "TotalWorkingYears"),
    ("Age", "TotalWorkingYears"),
    ("YearsAtCompany", "TotalWorkingYears"),
    ("DistanceFromHome", "MonthlyIncome")
]

filas_correlaciones = []

for variable_x, variable_y in pares_correlaciones:
    pares_validos = df[[variable_x, variable_y]].dropna()

    rho = pares_validos[variable_x].corr(
        pares_validos[variable_y],
        method="spearman"
    )

    filas_correlaciones.append({
        "variable_x": variable_x,
        "variable_y": variable_y,
        "n": len(pares_validos),
        "rho_spearman": rho
    })

referencias_correlaciones = pd.DataFrame(filas_correlaciones)

# Formatear únicamente la presentación.
presentacion_correlaciones = referencias_correlaciones.copy()

presentacion_correlaciones["rho_spearman"] = (
    presentacion_correlaciones["rho_spearman"].map(
        lambda valor: f"{valor:.4f}".replace(".", ",")
    )
)

display(presentacion_correlaciones)

ruta_correlaciones = (
    carpeta_resultados / "S1_referencia_correlaciones.csv"
)

referencias_correlaciones.to_csv(
    ruta_correlaciones,
    index=False,
    encoding="utf-8-sig",
    float_format="%.17g"
)

print("Referencia guardada:", ruta_correlaciones)

,variable_x,variable_y,n,rho_spearman
0,MonthlyIncome,TotalWorkingYears,1470,"0,7100"
1,Age,TotalWorkingYears,1470,"0,6569"
2,YearsAtCompany,TotalWorkingYears,1470,"0,5942"
3,DistanceFromHome,MonthlyIncome,1470,"0,0025"


Referencia guardada: C:\Users\Papá\Documents\2026\Magister\estadisticaComputacional\S2\resultados\S1_referencia_correlaciones.csv


## 3. Pruebas y proporciones de referencia de la Sumativa 1

Se reproducen chi-cuadrado sin corrección de continuidad para horas extra
y salida, y t de Welch bilateral para ingreso mensual según salida.

La diferencia de ingresos se define como permanece menos sale.
También se conservan los tamaños y las proporciones de salida por horas
extra como entradas para la simulación Monte Carlo.

In [4]:
# Chi-cuadrado: horas extra y salida.
contingencia = pd.crosstab(
    df["OverTime"], df["Attrition"]
).reindex(index=["No", "Yes"], columns=["No", "Yes"])

chi2, p_chi2, gl_chi2, esperadas = stats.chi2_contingency(
    contingencia,
    correction=False
)

v_cramer = np.sqrt(chi2 / contingencia.to_numpy().sum())

# Welch: ingreso de quienes permanecen menos ingreso de quienes salen.
ingreso_no = df.loc[
    df["Attrition"].eq("No"), "MonthlyIncome"
].dropna()

ingreso_si = df.loc[
    df["Attrition"].eq("Yes"), "MonthlyIncome"
].dropna()

welch = stats.ttest_ind(
    ingreso_no,
    ingreso_si,
    equal_var=False,
    alternative="two-sided"
)

ic_diferencia = welch.confidence_interval(confidence_level=0.95)

referencias_pruebas = pd.DataFrame([
    {
        "prueba": "Chi-cuadrado sin correccion",
        "estadistico": chi2,
        "gl": gl_chi2,
        "valor_p": p_chi2,
        "v_cramer": v_cramer,
        "esperada_minima": esperadas.min(),
        "diferencia_medias": np.nan,
        "ic95_inferior": np.nan,
        "ic95_superior": np.nan
    },
    {
        "prueba": "Welch bilateral permanece menos sale",
        "estadistico": welch.statistic,
        "gl": welch.df,
        "valor_p": welch.pvalue,
        "v_cramer": np.nan,
        "esperada_minima": np.nan,
        "diferencia_medias": ingreso_no.mean() - ingreso_si.mean(),
        "ic95_inferior": ic_diferencia.low,
        "ic95_superior": ic_diferencia.high
    }
])

# Entradas de Monte Carlo: tamaños y proporciones observadas por grupo.
referencias_grupos = pd.DataFrame({
    "n": contingencia.sum(axis=1),
    "salidas": contingencia["Yes"]
})

referencias_grupos["proporcion_salida"] = (
    referencias_grupos["salidas"] / referencias_grupos["n"]
)

referencias_grupos.index.name = "OverTime"
referencias_grupos = referencias_grupos.reset_index()

# Exportar valores sin redondearlos.
referencias_pruebas.to_csv(
    carpeta_resultados / "S1_referencia_pruebas.csv",
    index=False,
    encoding="utf-8-sig",
    float_format="%.17g"
)

referencias_grupos.to_csv(
    carpeta_resultados / "S1_referencia_grupos.csv",
    index=False,
    encoding="utf-8-sig",
    float_format="%.17g"
)

print(f"Chi-cuadrado: {chi2:.4f}; p = {p_chi2:.6g}")
print(f"V de Cramér: {v_cramer:.4f}")
print(f"Welch: t = {welch.statistic:.4f}; p = {welch.pvalue:.6g}")
print(
    "Diferencia media:",
    f"{ingreso_no.mean() - ingreso_si.mean():.2f}"
)
print(
    "IC del 95 %:",
    f"[{ic_diferencia.low:.2f}; {ic_diferencia.high:.2f}]"
)

display(referencias_grupos)

Chi-cuadrado: 89.0439; p = 3.86152e-21
V de Cramér: 0.2461
Welch: t = 7.4826; p = 4.43359e-13
Diferencia media: 2045.65
IC del 95 %: [1508.24; 2583.05]


,OverTime,n,salidas,proporcion_salida
0,No,1054,110,0.104364
1,Yes,416,127,0.305288


## 4. Registro de trazabilidad

Se registran las rutas, huellas SHA-256, versiones del entorno y archivos
de referencia. Las huellas permiten detectar cambios en el dataset
o en el notebook de S1.

Los cálculos anteriores reproducen los procedimientos de S1; su coincidencia
numérica es una comprobación de referencia, no una validación de robustez.
Esta última se realizará mediante bootstrap, permutación y sensibilidad.

In [5]:
archivos_referencia = [
    "S1_referencia_medias.csv",
    "S1_referencia_correlaciones.csv",
    "S1_referencia_pruebas.csv",
    "S1_referencia_grupos.csv"
]

registro = {
    "equipo": [
        "Hernán Saavedra",
        "Lenin Salas",
        "Rodrigo Gómez"
    ],
    "dataset": {
        "ruta": ruta_datos.relative_to(raiz).as_posix(),
        "sha256": huella_csv,
        "registros": len(df),
        "columnas": len(df.columns)
    },
    "notebook_s1": {
        "ruta": ruta_notebook_s1.relative_to(raiz).as_posix(),
        "sha256": huella_notebook_s1
    },
    "versiones": versiones,
    "semilla_base": SEMILLA,
    "iteraciones_previstas": N_ITERACIONES,
    "referencias": [
        {
            "ruta": (carpeta_resultados / nombre)
                .relative_to(raiz).as_posix(),
            "sha256": hashlib.sha256(
                (carpeta_resultados / nombre).read_bytes()
            ).hexdigest()
        }
        for nombre in archivos_referencia
    ]
}

ruta_registro = carpeta_resultados / "S1_trazabilidad.json"

ruta_registro.write_text(
    json.dumps(registro, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

print("Registro creado:", ruta_registro)

for referencia in registro["referencias"]:
    print(referencia["ruta"])

Registro creado: C:\Users\Papá\Documents\2026\Magister\estadisticaComputacional\S2\resultados\S1_trazabilidad.json
S2/resultados/S1_referencia_medias.csv
S2/resultados/S1_referencia_correlaciones.csv
S2/resultados/S1_referencia_pruebas.csv
S2/resultados/S1_referencia_grupos.csv
